# LLM Transformer Architecture

This notebook now contains complete worked solutions for the Topic 3 exercises on **Transformer Language Model Architecture**.

Each section includes:
- a short explanation of the concept being tested
- a concrete, executable reference implementation where appropriate
- direct answers to the written questions
- shape checks and sanity checks so the reasoning is verifiable in code

The emphasis is on implementation quality, tensor-shape discipline, and practical reasoning about optimization and resource cost.


In [ ]:
# Environment bootstrap for Google Colab and local notebooks.
# Run this cell first if the runtime is missing the assignment packages.

import sys
import subprocess

REQUIRED_PACKAGES = [
    "einops>=0.8",
    "einx>=0.4",
    "jaxtyping>=0.3",
    "numpy>=2.4",
    "psutil>=7",
    "pytest>=9.0",
    "pytest-timeout",
    "pypdf>=6.10.0",
    "regex>=2026.3.32",
    "tiktoken>=0.12.0",
    "torch~=2.11.0",
    "tqdm>=4.67",
    "wandb>=0.25",
    "ty>=0.0.26",
    "ruff>=0.15.8",
]

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *REQUIRED_PACKAGES])
else:
    print("Local runtime detected; using the currently selected kernel environment.")


In [ ]:
# Shared imports, deterministic setup, and reference implementations used below.

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

torch.manual_seed(0)

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)
print("Torch version:", torch.__version__)

def print_shape(name, tensor):
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")


def make_causal_mask(seq_len: int, device: torch.device | None = None) -> torch.Tensor:
    return torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    rotated = torch.stack((-x_odd, x_even), dim=-1)
    return rearrange(rotated, "... d pair -> ... (d pair)")


def apply_rope(x: torch.Tensor, positions: torch.Tensor | None = None, theta: float = 10000.0) -> torch.Tensor:
    """Apply rotary position embeddings over the last dimension."""
    d_head = x.shape[-1]
    if d_head % 2 != 0:
        raise ValueError("RoPE requires an even head dimension so features can be rotated in pairs.")

    seq_len = x.shape[-2]
    if positions is None:
        positions = torch.arange(seq_len, device=x.device)
    positions = positions.to(device=x.device, dtype=x.dtype)

    pair_idx = torch.arange(0, d_head, 2, device=x.device, dtype=x.dtype)
    inv_freq = 1.0 / (theta ** (pair_idx / d_head))
    angles = positions[..., None] * inv_freq

    cos = torch.repeat_interleave(torch.cos(angles), 2, dim=-1)
    sin = torch.repeat_interleave(torch.sin(angles), 2, dim=-1)
    while cos.ndim < x.ndim:
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)

    return x * cos + rotate_half(x) * sin


def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    d_k = q.shape[-1]
    scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, v)
    return output, scores, weights


class ReferenceLinear(nn.Module):
    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(d_out, d_in) / math.sqrt(d_in))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight.T


class ReferenceEmbedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(vocab_size, d_model) / math.sqrt(d_model))

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.weight[token_ids]


class ReferenceRMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x.pow(2), dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight


class ReferenceSwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = ReferenceLinear(d_model, d_ff)
        self.w2 = ReferenceLinear(d_ff, d_model)
        self.w3 = ReferenceLinear(d_model, d_ff)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.w1(x))
        value = self.w3(x)
        hidden = gate * value
        return self.w2(hidden)


class ReferenceCausalMultiheadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, theta: float = 10000.0, use_rope: bool = True):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must divide evenly by num_heads.")
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.theta = theta
        self.use_rope = use_rope

        self.q_proj = ReferenceLinear(d_model, d_model)
        self.k_proj = ReferenceLinear(d_model, d_model)
        self.v_proj = ReferenceLinear(d_model, d_model)
        self.o_proj = ReferenceLinear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        q = rearrange(self.q_proj(x), "b s (h d) -> b h s d", h=self.num_heads)
        k = rearrange(self.k_proj(x), "b s (h d) -> b h s d", h=self.num_heads)
        v = rearrange(self.v_proj(x), "b s (h d) -> b h s d", h=self.num_heads)

        if self.use_rope:
            q = apply_rope(q, theta=self.theta)
            k = apply_rope(k, theta=self.theta)

        seq_len = x.shape[1]
        mask = make_causal_mask(seq_len, x.device).view(1, 1, seq_len, seq_len)
        context, scores, weights = scaled_dot_product_attention(q, k, v, mask=mask)
        merged = rearrange(context, "b h s d -> b s (h d)")
        out = self.o_proj(merged)
        cache = {
            "q": q,
            "k": k,
            "v": v,
            "scores": scores,
            "weights": weights,
            "mask": mask,
            "merged": merged,
        }
        return out, cache


class ReferenceTransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, theta: float = 10000.0):
        super().__init__()
        self.norm1 = ReferenceRMSNorm(d_model)
        self.attn = ReferenceCausalMultiheadSelfAttention(d_model, num_heads, theta=theta, use_rope=True)
        self.norm2 = ReferenceRMSNorm(d_model)
        self.ffn = ReferenceSwiGLU(d_model, d_ff)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_input = self.norm1(x)
        attn_out, _ = self.attn(attn_input)
        x = x + attn_out
        ffn_input = self.norm2(x)
        x = x + self.ffn(ffn_input)
        return x


@dataclass
class TransformerLMConfig:
    vocab_size: int
    d_model: int
    num_layers: int
    num_heads: int
    d_ff: int
    theta: float = 10000.0


class ReferenceTransformerLM(nn.Module):
    def __init__(self, config: TransformerLMConfig):
        super().__init__()
        self.config = config
        self.token_embedding = ReferenceEmbedding(config.vocab_size, config.d_model)
        self.blocks = nn.ModuleList(
            [
                ReferenceTransformerBlock(
                    d_model=config.d_model,
                    num_heads=config.num_heads,
                    d_ff=config.d_ff,
                    theta=config.theta,
                )
                for _ in range(config.num_layers)
            ]
        )
        self.final_norm = ReferenceRMSNorm(config.d_model)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        x = self.token_embedding(token_ids)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = F.linear(x, self.token_embedding.weight)
        return logits


def count_parameters(module: nn.Module) -> int:
    return sum(param.numel() for param in module.parameters())


def bytes_to_mib(num_bytes: int) -> float:
    return num_bytes / (1024 ** 2)


## transformer_lm_overview

**Assignment description**

- Understand the full flow of Section 3.1 **Transformer LM**.
- Explain what token embeddings do in Section 3.1.0.1 **Token Embeddings**.
- Explain why the architecture is **pre-norm** in Section 3.1.0.2 **Pre-norm Transformer Block**.
- Track tensor shapes from token IDs to output logits.

**What this is testing**

This topic checks whether you can see the entire model as one coherent pipeline instead of a bag of independent modules. If you understand this section, later implementation bugs become much easier to localize.

**Pseudocode / attack plan**

```text
input token_ids: (batch, seq)
x = token_embedding[token_ids]                     # (batch, seq, d_model)
for each transformer block:
    h = rmsnorm_1(x)
    x = x + attention(h)
    h = rmsnorm_2(x)
    x = x + feed_forward(h)
x = final_rmsnorm(x)
logits = lm_head(x)                                # (batch, seq, vocab_size)
return logits
```

**Writeup guidance**

- Keep a one-line summary for the role of embeddings.
- Keep a one-line summary for why pre-norm helps optimization.
- Always annotate the shape after each major step.

In [ ]:
# transformer_lm_overview completed solution

batch_size = 2
seq_len = 4
vocab_size = 16
d_model = 8

token_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
embedding_table = torch.randn(vocab_size, d_model)
embedded = embedding_table[token_ids]
logits = torch.randn(batch_size, seq_len, vocab_size)

print_shape("token_ids", token_ids)
print_shape("embedded", embedded)
print_shape("logits", logits)

pipeline = [
    "1. token_ids: (batch, seq) = (2, 4)",
    "2. token embedding lookup -> x: (batch, seq, d_model) = (2, 4, 8)",
    "3. for each pre-norm block: RMSNorm keeps (2, 4, 8), attention returns (2, 4, 8), residual add keeps (2, 4, 8)",
    "4. second RMSNorm keeps (2, 4, 8), feed-forward returns (2, 4, 8), residual add keeps (2, 4, 8)",
    "5. final RMSNorm keeps (2, 4, 8)",
    "6. LM head projects the last dimension from d_model=8 to vocab_size=16, giving logits (2, 4, 16)",
]
print("Forward pipeline with exact shapes:")
for step in pipeline:
    print(step)

answer_embeddings = "Token embeddings map each discrete token ID to a learned dense vector, so the model can operate in a continuous representation space."
answer_pre_norm = "The architecture is pre-norm because normalizing before each sublayer stabilizes residual updates and makes deep optimization easier."
answer_residual = "Residual connections appear after attention and after the feed-forward sublayer; they preserve shape because both branches produce tensors with the same (batch, seq, d_model) shape as the input."
answer_logits = "The final dimension of the logits is vocab_size because each token position needs one score per vocabulary item before softmax."

print(answer_embeddings)
print(answer_pre_norm)
print(answer_residual)
print(answer_logits)

toy_model = ReferenceTransformerLM(
    TransformerLMConfig(vocab_size=vocab_size, d_model=d_model, num_layers=2, num_heads=2, d_ff=16)
)
toy_logits = toy_model(token_ids)
print_shape("toy_logits", toy_logits)
assert toy_logits.shape == (batch_size, seq_len, vocab_size)


## batching_einsum_and_memory_ordering

**Assignment description**

- Work through Section 3.2 **Remark: Batching, Einsum and Efficient Computation**.
- Understand Section 3.2.1 **Mathematical Notation and Memory Ordering**.
- Rewrite nested-loop tensor operations as batched matrix operations.

**What this is testing**

This topic checks whether you can translate equations into efficient tensor code. Most architecture bugs in this assignment are really shape bugs, transpose bugs, or broadcasting bugs.

**Pseudocode / attack plan**

```text
start with a scalar formula
identify batch dimensions
identify sequence dimensions
identify feature dimensions
group dimensions so one matmul handles many examples at once
verify the output shape before writing the code
only then choose between matmul, einsum, or reshape+matmul
```

**Writeup guidance**

- State clearly which dimensions are batch-like and which are feature-like.
- If you use `rearrange`, explain what semantic change the reshape is expressing.

In [ ]:
# batching_einsum_and_memory_ordering completed solution

x = torch.randn(2, 3, 4)
W = torch.randn(5, 4)

y = x @ W.T
y_einsum = torch.einsum("bsd,od->bso", x, W)

print_shape("x", x)
print_shape("W", W)
print_shape("y", y)
assert torch.allclose(y, y_einsum)

q = torch.randn(2, 3, 6, 8)
k = torch.randn(2, 3, 6, 8)
scores = torch.matmul(q, k.transpose(-1, -2))
scores_einsum = torch.einsum("bhqd,bhkd->bhqk", q, k)
print_shape("scores", scores)
assert torch.allclose(scores, scores_einsum)

answer_einsum = "A correct einsum form for the linear layer is einsum('bsd,od->bso', x, W): batch b and sequence s are batch-like dimensions, d is contracted, and o is the new output feature dimension."
answer_scores = "Attention scores end with (queries, keys) because each query position produces one compatibility score against every key position, so the last two axes are 'which query asked?' and 'which key was compared?'."
answer_transpose = "A dangerous transpose bug is using q.transpose(-1, -2) @ k or q @ k without transposing the key sequence axis; if dimensions happen to align, the code may run while contracting the wrong semantic axes and producing meaningless scores."

print(answer_einsum)
print(answer_scores)
print(answer_transpose)


## parameter_initialization

**Assignment description**

- Study Section 3.3.1 **Parameter Initialization**.
- Explain why initialization scale matters for deep residual networks.
- Connect bad initialization to exploding activations, vanishing activations, or unstable gradients.

**What this is testing**

This topic checks whether you understand that a model can be mathematically correct and still fail immediately because the parameter scale is wrong.

**Pseudocode / attack plan**

```text
choose a tensor shape
sample weights from a proposed initialization rule
push a random batch through the layer
measure output mean and variance
repeat across stacked layers
compare stable vs unstable scales
```

**Writeup guidance**

- Do not memorize formulas blindly.
- Focus on the practical question: does the signal stay at a reasonable scale through depth?

In [ ]:
# parameter_initialization completed solution

d_in = 512
d_out = 512
x = torch.randn(64, d_in)

W_small = 0.01 * torch.randn(d_out, d_in)
W_large = 1.00 * torch.randn(d_out, d_in)

y_small = x @ W_small.T
y_large = x @ W_large.T

print("single-layer small std:", y_small.std().item())
print("single-layer large std:", y_large.std().item())


def residual_stack_stds(scale: float, depth: int = 6) -> list[float]:
    h = torch.randn(256, d_in)
    stds = [h.std().item()]
    for _ in range(depth):
        W = scale * torch.randn(d_out, d_in)
        h = h + h @ W.T
        stds.append(h.std().item())
    return stds


scales = {
    "too_small": 0.01,
    "balanced_1/sqrt(d_in)": 1 / math.sqrt(d_in),
    "too_large": 1.0,
}

for name, scale in scales.items():
    stds = residual_stack_stds(scale)
    print(f"{name} residual stds by depth: {[round(v, 3) for v in stds]}")

answer_scale = "Tiny initialization keeps residual updates weak and the layer behaves almost like the identity; a balanced scale keeps activations in a usable range; a large scale makes activations explode within a few layers."
answer_residual = "Residual stacks are sensitive because every block adds a new transformed branch back into the running hidden state, so mis-scaled branches accumulate across depth instead of staying local to one layer."
answer_training = "Initialization directly affects training stability: if activations or gradients start too large, optimization becomes noisy or diverges; if they start too small, the network learns slowly because each residual branch contributes almost no useful signal."

print(answer_scale)
print(answer_residual)
print(answer_training)


## linear_module

**Assignment description**

- Implement Section 3.3.2 **Linear Module**.
- Match the behavior expected by `tests/test_model.py::test_linear`.
- Make sure your implementation works with arbitrary leading dimensions.

**What this is testing**

This is the simplest core module in the architecture, but it establishes the exact convention used everywhere else: weight shape, broadcasting behavior, and output shape semantics.

**Pseudocode / attack plan**

```text
input x has shape (..., d_in)
weight W has shape (d_out, d_in)
compute y = x @ W^T
return tensor with shape (..., d_out)
```

**Implementation notes**

- The tests pass batched tensors, so do not flatten away leading dimensions.
- If you use `torch.matmul`, the leading dimensions are handled automatically.

In [ ]:
# linear_module completed solution

x = torch.randn(2, 3, 4)
W = torch.randn(6, 4)
y = x @ W.T

print_shape("x", x)
print_shape("W", W)
print_shape("y", y)

linear = ReferenceLinear(d_in=4, d_out=6)
with torch.no_grad():
    linear.weight.copy_(W)
module_y = linear(x)
assert torch.allclose(module_y, y)

answer_weight_shape = "The stored weight shape is (d_out, d_in) so each row is one output neuron's weight vector over the full input feature dimension."
answer_transpose = "The exact transpose in the forward pass is y = x @ W.T, which changes W from (d_out, d_in) to (d_in, d_out) so the last dimension of x can contract with d_in."
answer_behavior = "The reference Linear module above works with arbitrary leading dimensions because matmul treats everything before the final feature dimension as batch dimensions automatically."

print(answer_weight_shape)
print(answer_transpose)
print(answer_behavior)


## embedding_module

**Assignment description**

- Implement Section 3.3.3 **Embedding Module**.
- Connect it back to Section 3.1.0.1 **Token Embeddings**.
- Match `tests/test_model.py::test_embedding`.

**What this is testing**

This checks whether you understand embeddings as table lookup rather than dense arithmetic. Token IDs are discrete indices, and the embedding layer converts them into continuous vectors.

**Pseudocode / attack plan**

```text
weights has shape (vocab_size, d_model)
token_ids has shape (...)
for each token id, fetch weights[token_id]
return tensor of shape (..., d_model)
```

**Writeup guidance**

- Emphasize that no one-hot matrix is needed in the efficient implementation.
- Track the output shape carefully.

In [ ]:
# embedding_module completed solution

vocab_size = 10
d_model = 6
token_ids = torch.tensor([[0, 3, 4], [2, 9, 1]])
embedding_weight = torch.randn(vocab_size, d_model)
embedded = embedding_weight[token_ids]

print_shape("token_ids", token_ids)
print_shape("embedded", embedded)

one_hot = F.one_hot(token_ids, num_classes=vocab_size).float()
one_hot_embedded = one_hot @ embedding_weight
assert torch.allclose(embedded, one_hot_embedded)

embedding = ReferenceEmbedding(vocab_size=vocab_size, d_model=d_model)
with torch.no_grad():
    embedding.weight.copy_(embedding_weight)
module_embedded = embedding(token_ids)
assert torch.allclose(module_embedded, embedded)

answer_one_hot = "Indexing is equivalent to multiplying by a one-hot vector because a one-hot row selects exactly one row of the embedding matrix and zeros out every other row."
answer_updates = "Embeddings are updated during backpropagation: only the rows corresponding to token IDs that appeared in the batch receive gradient contributions on that step."
answer_behavior = "The efficient implementation uses direct table lookup instead of explicitly materializing one-hot vectors, but it produces the same result."

print(answer_one_hot)
print(answer_updates)
print(answer_behavior)


## rmsnorm

**Assignment description**

- Implement Section 3.4.1 **Root Mean Square Layer Normalization**.
- Match `tests/test_model.py::test_rmsnorm`.
- Explain how RMSNorm differs from LayerNorm.

**What this is testing**

This exercise checks whether you can normalize activations along the feature dimension while keeping the implementation numerically stable and shape-correct.

**Pseudocode / attack plan**

```text
input x: (..., d_model)
rms = sqrt(mean(x^2 over last dim) + eps)
normalized = x / rms
output = normalized * learned_weight
return output
```

**Implementation notes**

- The mean is taken over the final feature dimension only.
- Keep `eps` inside the square root for stability.

In [ ]:
# rmsnorm completed solution

x = torch.randn(2, 4, 8)
weight = torch.ones(8)
eps = 1e-5

rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + eps)
y = (x / rms) * weight

print_shape("x", x)
print_shape("rms", rms)
print_shape("y", y)

rmsnorm = ReferenceRMSNorm(d_model=8, eps=eps)
with torch.no_grad():
    rmsnorm.weight.copy_(weight)
module_y = rmsnorm(x)
assert torch.allclose(module_y, y)

layernorm = nn.LayerNorm(8, elementwise_affine=False)
layernorm_y = layernorm(x)
print("mean after RMSNorm (not forced to zero):", y.mean(dim=-1))
print("mean after LayerNorm (~zero):", layernorm_y.mean(dim=-1))

answer_mean = "RMSNorm keeps the mean information because it does not subtract the feature-wise average; it only rescales each token vector by its root-mean-square magnitude."
answer_compare = "Unlike standard LayerNorm, RMSNorm divides by RMS without centering the activations, so it normalizes scale but not mean."
answer_behavior = "The reference RMSNorm module above matches the formula exactly: normalize over the last dimension, keep epsilon inside the square root, and multiply by a learned per-feature weight."

print(answer_mean)
print(answer_compare)
print(answer_behavior)


## positionwise_feedforward_network

**Assignment description**

- Implement Section 3.4.2 **Position-Wise Feed-Forward Network**.
- In this codebase, the feed-forward block is tested via `run_swiglu`.
- Understand how the nonlinearity acts independently at each sequence position.

**What this is testing**

This checks whether you understand the FFN as a per-token transformation, not a sequence-mixing operation. The same weights are applied at every position.

**Pseudocode / attack plan**

```text
input x: (..., d_model)
gate = silu(x @ W1^T)
value = x @ W3^T
hidden = gate * value
output = hidden @ W2^T
return output
```

**Writeup guidance**

- Make explicit that there is no attention or token interaction inside this sublayer.
- Track shapes for `d_model -> d_ff -> d_model`.

In [ ]:
# positionwise_feedforward_network completed solution

batch, seq, d_model, d_ff = 2, 4, 8, 16
x = torch.randn(batch, seq, d_model)
w1 = torch.randn(d_ff, d_model)
w2 = torch.randn(d_model, d_ff)
w3 = torch.randn(d_ff, d_model)

gate = torch.nn.functional.silu(x @ w1.T)
value = x @ w3.T
hidden = gate * value
y = hidden @ w2.T

print_shape("hidden", hidden)
print_shape("y", y)

ffn = ReferenceSwiGLU(d_model=d_model, d_ff=d_ff)
with torch.no_grad():
    ffn.w1.weight.copy_(w1)
    ffn.w2.weight.copy_(w2)
    ffn.w3.weight.copy_(w3)
module_y = ffn(x)
assert torch.allclose(module_y, y)

answer_positionwise = "This module is called position-wise because each token position is transformed independently with the same weights; there is no mixing across the sequence axis."
answer_gating = "In SwiGLU, the gate path learns how much of each hidden feature should pass through, while the value path carries the candidate content that gets multiplicatively filtered."
answer_shapes = "The shape flow is (batch, seq, d_model) -> (batch, seq, d_ff) -> (batch, seq, d_model), so capacity expands inside the block and contracts back to the model width."

print(answer_positionwise)
print(answer_gating)
print(answer_shapes)


## relative_positional_embeddings

**Assignment description**

- Implement Section 3.4.3 **Relative Positional Embeddings**.
- In this assignment, that means implementing **RoPE** and matching `tests/test_model.py::test_rope`.
- Understand why RoPE is applied to queries and keys, not values.

**What this is testing**

This topic checks whether you can take a mathematically unusual positional encoding scheme and implement it without losing track of shapes, pairwise dimensions, or broadcasting.

**Pseudocode / attack plan**

```text
split the last dimension into even/odd pairs
compute inverse frequencies from theta and pair index
for each token position, build rotation angles
rotate each even/odd pair by that angle
recombine into the original last dimension
```

**Writeup guidance**

- State clearly that RoPE changes representation as a function of token position.
- Mention that attention now depends on relative offsets through rotated Q and K vectors.

In [ ]:
# relative_positional_embeddings completed solution

d_k = 8
theta = 10000.0
seq_len = 6
x = torch.randn(1, seq_len, d_k)
positions = torch.arange(seq_len).unsqueeze(0)

pair_idx = torch.arange(0, d_k, 2)
inv_freq = 1.0 / (theta ** (pair_idx / d_k))
angles = positions[..., None] * inv_freq
x_rope = apply_rope(x, positions=positions.squeeze(0), theta=theta)

print_shape("x", x)
print_shape("positions", positions)
print_shape("angles", angles)
print_shape("x_rope", x_rope)
print("RoPE preserves per-token vector norms:", torch.allclose(x.norm(dim=-1), x_rope.norm(dim=-1), atol=1e-5))
print("Position 0 is unchanged because its rotation angle is zero:", torch.allclose(x[:, 0], x_rope[:, 0], atol=1e-5))

answer_pairs = "The last dimension must be split into even/odd pairs because RoPE performs a 2D rotation on each feature pair, analogous to rotating coordinates in a plane."
answer_before_qk = "RoPE is applied before the QK dot product so the position-dependent rotation is already baked into the query and key vectors whose inner product defines attention scores."
answer_relative = "Because both Q and K are rotated as a function of position, their dot product depends on relative offsets, which is why RoPE encodes relative position information without adding separate bias tensors."

print(answer_pairs)
print(answer_before_qk)
print(answer_relative)


## scaled_dot_product_attention

**Assignment description**

- Implement Section 3.4.4 **Scaled Dot-Product Attention**.
- Match both `test_scaled_dot_product_attention` and `test_4d_scaled_dot_product_attention`.
- Support optional masking.

**What this is testing**

This is the mathematical core of attention. The exercise checks shape reasoning, masking semantics, numerical stability, and the ability to generalize from 2D examples to batched multi-head tensors.

**Pseudocode / attack plan**

```text
scores = Q @ K^T
scores = scores / sqrt(d_k)
if mask exists:
    set masked positions to a very negative value
weights = softmax(scores, dim=-1)
output = weights @ V
return output
```

**Implementation notes**

- The softmax dimension is over keys.
- The function must preserve all leading batch/head dimensions.

In [ ]:
# scaled_dot_product_attention completed solution

Q = torch.randn(2, 3, 4, 8)
K = torch.randn(2, 3, 4, 8)
V = torch.randn(2, 3, 4, 8)
mask = torch.tril(torch.ones(4, 4, dtype=torch.bool)).view(1, 1, 4, 4)

scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(Q.shape[-1])
scores = scores.masked_fill(~mask, float("-inf"))
weights = torch.softmax(scores, dim=-1)
output = torch.matmul(weights, V)

print_shape("scores", scores)
print_shape("weights", weights)
print_shape("output", output)

module_output, module_scores, module_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
assert torch.allclose(output, module_output, equal_nan=True)
assert torch.allclose(weights, module_weights, equal_nan=True)

answer_scale = "The scaling factor uses sqrt(d_k) because raw dot products grow in variance with head dimension; dividing by sqrt(d_k) keeps score magnitudes in a range where softmax does not become prematurely saturated."
answer_softmax = "If you softmax over the wrong dimension, you normalize across queries or heads instead of across candidate keys, so the weights stop meaning 'how much should this query attend to each key?'."
answer_behavior = "The reference SDPA function preserves all leading batch and head dimensions and only treats the final key axis as the softmax axis, which is the correct attention semantics."

print(answer_scale)
print(answer_softmax)
print(answer_behavior)


## causal_multihead_self_attention

**Assignment description**

- Implement Section 3.4.5 **Causal Multi-Head Self-Attention**.
- Include Section 3.4.5.1 **Causal masking**.
- Include Section 3.4.5.2 **Applying RoPE**.
- Match both `test_multihead_self_attention` and `test_multihead_self_attention_with_rope`.

**What this is testing**

This exercise checks whether you can compose multiple correct subroutines into one attention layer: projections, head splitting, masking, positional rotation, head concatenation, and output projection.

**Pseudocode / attack plan**

```text
project input x into Q, K, V using one linear map each
reshape from (..., seq, d_model) to (..., heads, seq, d_head)
optionally apply RoPE to Q and K
build a causal mask so token i cannot see tokens > i
run scaled dot-product attention independently per head
transpose/reshape heads back into (..., seq, d_model)
apply output projection
return result
```

**Implementation notes**

- `d_model` must divide evenly by `num_heads`.
- Causal masking happens in score space before softmax.
- RoPE is applied per head in `d_head`, not across the full `d_model`.


In [ ]:
# causal_multihead_self_attention completed solution

batch, seq, d_model, num_heads = 2, 5, 12, 3
d_head = d_model // num_heads
x = torch.randn(batch, seq, d_model)

attn = ReferenceCausalMultiheadSelfAttention(d_model=d_model, num_heads=num_heads, use_rope=True)
out, cache = attn(x)

print_shape("q", cache["q"])
print_shape("scores", cache["scores"])
print_shape("merged", cache["merged"])
print_shape("out", out)
assert out.shape == (batch, seq, d_model)

perturbed = x.clone()
perturbed[:, -1] += 1000.0
out_perturbed, _ = attn(perturbed)
print("Earlier positions are unchanged by a future-token perturbation:", torch.allclose(out[:, :-1], out_perturbed[:, :-1], atol=1e-5))

answer_structure = "The real layer first projects x into Q, K, and V, then splits into heads, applies RoPE to Q and K, applies causal masking in score space, merges heads back together, and finally applies the output projection."
answer_rope = "RoPE is inserted after the Q and K projections are reshaped into per-head tensors and before the QK^T dot product, because the rotation acts inside each head dimension."
answer_masking = "Causal masking must happen before softmax so illegal future positions receive effectively zero probability mass; masking after softmax would renormalize nothing and still let future tokens influence the output."

print(answer_structure)
print(answer_rope)
print(answer_masking)


## pre_norm_transformer_block

**Assignment description**

- Implement the full Section 3.4 **Pre-Norm Transformer Block**.
- Combine RMSNorm, causal MHA with RoPE, residual addition, and feed-forward.
- Match `tests/test_model.py::test_transformer_block`.

**What this is testing**

This is the first place where architectural composition matters more than any single formula. A correct block needs the right submodules and the right ordering.

**Pseudocode / attack plan**

```text
input x
attn_input = rmsnorm_1(x)
x = x + mha_with_rope(attn_input)
ffn_input = rmsnorm_2(x)
x = x + ffn(ffn_input)
return x
```

**Writeup guidance**

- The order matters: norm first, sublayer second, residual add third.
- Note that both residual branches preserve `(batch, seq, d_model)`.

In [ ]:
# pre_norm_transformer_block completed solution

x = torch.randn(2, 5, 8)

block = ReferenceTransformerBlock(d_model=8, num_heads=2, d_ff=16)
block_out = block(x)

attn_input = block.norm1(x)
attn_out, _ = block.attn(attn_input)
after_attn = x + attn_out
ffn_input = block.norm2(after_attn)
after_ffn = after_attn + block.ffn(ffn_input)

print_shape("x", x)
print_shape("after_attn", after_attn)
print_shape("after_ffn", after_ffn)
assert torch.allclose(block_out, after_ffn)

execution_order = [
    "1. attn_input = rmsnorm_1(x)",
    "2. attn_out = mha_with_rope(attn_input)",
    "3. x = x + attn_out",
    "4. ffn_input = rmsnorm_2(x)",
    "5. ffn_out = swiglu(ffn_input)",
    "6. x = x + ffn_out",
]
print("Execution order:")
for step in execution_order:
    print(step)

parameter_groups = [
    "RMSNorm 1 scale parameter",
    "Attention Q, K, V, and output projection weights",
    "RMSNorm 2 scale parameter",
    "SwiGLU weights W1, W2, and W3",
]
print("Learned parameter groups:")
for group in parameter_groups:
    print("-", group)


## full_transformer_lm_and_resource_accounting

**Assignment description**

- Implement Section 3.5 **The Full Transformer LM**.
- Work through Section 3.5.0.1 **Resource accounting**.
- Match `tests/test_model.py::test_transformer_lm` and `test_transformer_lm_truncated_input`.
- Estimate parameter count and activation growth as a function of model size.

**What this is testing**

This final Section 3 topic checks whether you can assemble the full model correctly and reason about engineering cost, not just mathematical correctness.

**Pseudocode / attack plan**

```text
embed tokens
run N transformer blocks in sequence
apply final rmsnorm
project to vocabulary logits with lm_head

for resource accounting:
count parameters in embeddings
count parameters in one block
multiply block count by number of layers
add final norm and lm head
estimate activation sizes from batch, seq, d_model, heads, and d_ff
```

**Writeup guidance**

- Keep the implementation checklist separate from the accounting checklist.
- When counting parameters, write the formula first and plug in numbers second.

In [ ]:
# full_transformer_lm_and_resource_accounting completed solution

vocab_size = 10000
d_model = 512
num_layers = 8
num_heads = 8
d_ff = 1344

embedding_params = vocab_size * d_model
attn_params_per_block = 4 * d_model * d_model
ffn_params_per_block = 3 * d_ff * d_model
norm_params_per_block = 2 * d_model
block_params = attn_params_per_block + ffn_params_per_block + norm_params_per_block
final_norm_params = d_model
untied_lm_head_params = vocab_size * d_model
tied_lm_head_params = 0

total_tied = embedding_params + num_layers * block_params + final_norm_params + tied_lm_head_params
total_untied = embedding_params + num_layers * block_params + final_norm_params + untied_lm_head_params

print("embedding_params:", embedding_params)
print("block_params:", block_params)
print("total_tied:", total_tied)
print("total_untied:", total_untied)

cfg = TransformerLMConfig(
    vocab_size=vocab_size,
    d_model=d_model,
    num_layers=num_layers,
    num_heads=num_heads,
    d_ff=d_ff,
)
model = ReferenceTransformerLM(cfg)
actual_params = count_parameters(model)
print("actual model params (tied head):", actual_params)
assert actual_params == total_tied

sample_token_ids = torch.randint(0, vocab_size, (2, 16))
logits = model(sample_token_ids)
print_shape("sample_token_ids", sample_token_ids)
print_shape("logits", logits)
assert logits.shape == (2, 16, vocab_size)

seq_for_accounting = 1024
score_elements = 2 * num_heads * seq_for_accounting * seq_for_accounting
score_bytes_fp32 = score_elements * 4
print("attention score elements for batch=2, heads=8, seq=1024:", score_elements)
print("attention score tensor size in MiB (fp32):", round(bytes_to_mib(score_bytes_fp32), 2))

answer_head = "This implementation ties the LM head to the token embedding matrix, so the output projection reuses embedding weights and adds zero new parameters; an untied head would add vocab_size * d_model extra parameters."
answer_activation = "A major activation tensor is the attention score matrix with shape (batch, heads, seq, seq); its quadratic growth in sequence length is one of the main memory bottlenecks in Transformer training."
answer_checklist = "The full model pipeline is token embedding -> N pre-norm Transformer blocks -> final RMSNorm -> vocabulary projection, and the executable reference model above verifies that this assembly produces logits with shape (batch, seq, vocab_size)."

print(answer_head)
print(answer_activation)
print(answer_checklist)
